# 3. Prediksi Harga 6 Bulan
Memprediksi harga penutupan bulanan tiap komoditas (rata-rata semua pasar). Lima model dibandingkan lewat uji mundur 24 bulan terakhir memakai MAPE dan RMSE: linear (Least Square) 36 bulan, polinomial derajat 2, Theta, Holt-Winters, dan SARIMA(1,1,0)(1,1,0)12. Model dengan MAPE terkecil dipakai untuk prediksi 6 bulan ke depan beserta rentang ketidakpastiannya.

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

data_path = '../data/processed/data_bersih.csv'
output_dir = '../data/processed'
item_grafik = 'rice'
bulan_depan = 6
uji_bulan = 24

In [ ]:
KOL_PASAR = 'mkt_name'
KOL_TANGGAL = 'date'
KOL_INDEKS = 'food_price_index'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

BATAS_WASPADA = 5
BATAS_LONJAKAN = 10
NAMA_KELAS = ['Normal', 'Waspada', 'Lonjakan']
FITUR_DASAR = ['inflasi_ini', 'inflasi_lalu', 'bulan']
PERSEN_TEST = 0.2
TAHUN_ACUAN_AKHIR = 2019

BATAS_HARGA_PERSEN = 5

NAMA_KOMODITAS = {
    'eggs': 'Telur', 'garlic': 'Bawang Putih', 'meat_beef': 'Daging Sapi', 'meat_chicken': 'Daging Ayam',
    'meat_chicken_broiler': 'Daging Ayam Broiler', 'oil': 'Minyak Goreng', 'onions': 'Bawang Merah',
    'rice': 'Beras', 'sugar': 'Gula', 'food_price_index': 'Indeks Pangan',
}
MUSIM_ONI = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']

In [ ]:
def model_regresi(y, h, derajat, jendela=36):
    y = y[-jendela:]
    X = np.arange(len(y)).reshape(-1, 1)
    model = make_pipeline(PolynomialFeatures(degree=derajat), LinearRegression()).fit(X, y)
    return model.predict(np.arange(len(y), len(y) + h).reshape(-1, 1))


MUSIM = 12


def model_theta(y, h):
    n = len(y)
    kemiringan = np.polyfit(np.arange(n), y, 1)[0]

    def ses(z, a):
        level = z[0]
        for v in z[1:]:
            level = a * v + (1 - a) * level
        return level

    a = min([0.2, 0.4, 0.6, 0.8], key=lambda a: sum((y[i] - ses(y[:i], a)) ** 2 for i in range(n - 12, n)))
    return ses(y, a) + 0.5 * kemiringan * (np.arange(1, h + 1) - 1 + 1 / a - ((1 - a) ** n) / a)


def latih_holt_winters(y, a, b, g):
    level, tren = y[:MUSIM].mean(), (y[MUSIM:2 * MUSIM].mean() - y[:MUSIM].mean()) / MUSIM
    musim, sse = list(y[:MUSIM] - level), 0.0
    for i in range(MUSIM, len(y)):
        galat = y[i] - (level + tren + musim[i - MUSIM])
        sse += galat * galat
        level_lalu = level
        level = a * (y[i] - musim[i - MUSIM]) + (1 - a) * (level + tren)
        tren = b * (level - level_lalu) + (1 - b) * tren
        musim.append(g * (y[i] - level) + (1 - g) * musim[i - MUSIM])
    return sse, level, tren, musim


def model_holt_winters(y, h):
    y = y[-96:]
    if len(y) < 2 * MUSIM + 6:
        return np.full(h, np.nan)
    terbaik = None
    for a in (0.2, 0.5, 0.8):
        for b in (0.01, 0.1):
            for g in (0.1, 0.3, 0.6):
                hasil = latih_holt_winters(y, a, b, g)
                if terbaik is None or hasil[0] < terbaik[0]:
                    terbaik = hasil
    _, level, tren, musim = terbaik
    n = len(y)
    return np.array([level + (k + 1) * tren + musim[n - MUSIM + (k % MUSIM)] for k in range(h)])


def model_sarima(y, h):
    # SARIMA(1,1,0)(1,1,0)12: data dibedakan sekali biasa dan sekali musiman, lalu diregresikan pada lag 1, 12, dan 13
    y = y[-120:]
    if len(y) < 4 * MUSIM:
        return np.full(h, np.nan)
    w = y[MUSIM + 1:] - y[MUSIM:-1] - y[1:-MUSIM] + y[:-MUSIM - 1]
    lag = [1, MUSIM, MUSIM + 1]
    X = np.column_stack([w[MUSIM + 1 - k:len(w) - k] for k in lag])
    koef = np.linalg.lstsq(X, w[MUSIM + 1:], rcond=None)[0]
    dw, dy, hasil = list(w), list(y), []
    for _ in range(h):
        w_baru = sum(koef[j] * dw[-lag[j]] for j in range(3))
        dw.append(w_baru)
        y_baru = dy[-1] + dy[-MUSIM] - dy[-MUSIM - 1] + w_baru
        dy.append(y_baru)
        hasil.append(y_baru)
    return np.array(hasil)


def kandidat_prediksi():
    return {
        'Linear (36 bulan)': lambda y, h: model_regresi(y, h, 1),
        'Polinomial derajat 2 (36 bulan)': lambda y, h: model_regresi(y, h, 2),
        'Theta': model_theta,
        'Holt-Winters': model_holt_winters,
        'SARIMA (1,1,0)(1,1,0)12': model_sarima,
    }


def prediksi_harga(seri, bulan_depan=6, uji_bulan=24):
    s = seri.dropna().sort_index().asfreq('MS').interpolate(limit_direction='both')
    y = s.values.astype(float)
    if len(y) < uji_bulan + bulan_depan + 12:
        return None
    kandidat = kandidat_prediksi()
    galat = {nama: [] for nama in kandidat}
    kuadrat = {nama: [] for nama in kandidat}
    for o in range(len(y) - uji_bulan - bulan_depan + 1, len(y) - bulan_depan + 1):
        aktual = y[o:o + bulan_depan]
        for nama, f in kandidat.items():
            pred = f(y[:o], bulan_depan)
            galat[nama].append(np.abs((aktual - pred) / aktual))
            kuadrat[nama].append((aktual - pred) ** 2)
    galat = {nama: np.array(v) for nama, v in galat.items()}
    tabel = pd.DataFrame({'MAPE (%)': {nama: v.mean() * 100 for nama, v in galat.items()},
                          'RMSE (Rp/kg)': {nama: np.sqrt(np.mean(v)) for nama, v in kuadrat.items()}})
    terbaik = tabel['MAPE (%)'].idxmin()
    lebar = np.maximum.accumulate(np.quantile(galat[terbaik], 0.9, axis=0))
    tanggal = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=bulan_depan, freq='MS')
    pred = kandidat[terbaik](y, bulan_depan)
    return {'model': terbaik, 'mape': float(tabel.loc[terbaik, 'MAPE (%)']), 'rmse': float(tabel.loc[terbaik, 'RMSE (Rp/kg)']), 'tabel': tabel.round(2),
            'prediksi': pd.Series(pred, index=tanggal), 'bawah': pd.Series(pred * (1 - lebar), index=tanggal),
            'atas': pd.Series(pred * (1 + lebar), index=tanggal)}

## Data bulanan
Setiap bulan dirata-ratakan dari semua pasar. Kolom c_ adalah harga penutupan bulan itu.

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
df['bulan'] = df[KOL_TANGGAL].dt.to_period('M').dt.to_timestamp()
print(f'Data: {df["bulan"].min():%b %Y} sampai {df["bulan"].max():%b %Y}')

## Uji mundur semua komoditas
Untuk tiap bulan dari 24 bulan terakhir, model dilatih memakai data sebelum bulan itu lalu diminta menebak 6 bulan berikutnya. MAPE adalah rata-rata kesalahan persentase.

In [ ]:
hasil_semua = {}
baris = []
baris_rmse = []
for item in KOMODITAS:
    seri = df.groupby('bulan')[f'c_{item}'].mean()
    hasil = prediksi_harga(seri, bulan_depan, uji_bulan)
    hasil_semua[item] = (seri, hasil)
    baris.append({'Komoditas': item, 'Model terpilih': hasil['model'], 'MAPE terpilih (%)': hasil['mape']}
                 | {nama: v for nama, v in hasil['tabel']['MAPE (%)'].items()})
    baris_rmse.append({'Komoditas': item} | {nama: v for nama, v in hasil['tabel']['RMSE (Rp/kg)'].items()})
ringkas = pd.DataFrame(baris).set_index('Komoditas')
ringkas.round(2)

In [ ]:
ringkas_rmse = pd.DataFrame(baris_rmse).set_index('Komoditas')
ringkas_rmse.round(0)

Tidak semua model unggul di semua komoditas, sehingga pemilihan dilakukan per komoditas lewat uji mundur. Holt-Winters dan SARIMA, yang unggul pada data wisata bermusim, tidak otomatis unggul di sini karena pola musiman tahunan harga pangan lemah. Tabel kedua memuat RMSE dalam Rp/kg.

In [ ]:
ringkas['Model terpilih'].value_counts()

## Candlestick dan prediksi
Satu batang per bulan: badan batang dari harga awal ke harga akhir, garis tipis dari harga terendah ke tertinggi. Hijau berarti harga akhir lebih tinggi dari harga awal.

In [ ]:
seri, hasil = hasil_semua[item_grafik]
ohlc = df.groupby('bulan')[[f'{p}_{item_grafik}' for p in 'ohlc']].mean().tail(48)
ohlc.columns = list('ohlc')
fig, ax = plt.subplots(figsize=(13, 6))
for tgl, r in ohlc.iterrows():
    warna = '#2e7d32' if r['c'] >= r['o'] else '#c62828'
    ax.vlines(tgl, r['l'], r['h'], color=warna, linewidth=1)
    ax.bar(tgl, abs(r['c'] - r['o']), bottom=min(r['o'], r['c']), width=20, color=warna)
ax.plot(hasil['prediksi'].index, hasil['prediksi'].values, color='#1565c0', linestyle='dashed', marker='o', markersize=3, label=f'Prediksi ({hasil["model"]})')
ax.fill_between(hasil['prediksi'].index, hasil['bawah'].values, hasil['atas'].values, color='#1565c0', alpha=0.2, label='Rentang ketidakpastian')
ax.set_title(f'Harga {NAMA_KOMODITAS[item_grafik]}: candlestick bulanan dan prediksi {bulan_depan} bulan')
ax.set_xlabel('Bulan')
ax.set_ylabel('Harga (IDR/kg)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Simpan hasil

In [ ]:
ringkas.to_csv(f'{output_dir}/metrik_prediksi_harga.csv')
forecast = pd.concat({item: pd.DataFrame({'prediksi': h['prediksi'], 'bawah': h['bawah'], 'atas': h['atas']}) for item, (s, h) in hasil_semua.items()},
                     names=['komoditas', 'bulan']).round(0)
forecast.to_csv(f'{output_dir}/hasil_prediksi_harga.csv')
print('Metrik dan prediksi tersimpan di', output_dir)